In [1]:
import pandas as pd
from pathlib import Path

races_dir = Path("../data/processed/races")
files = sorted(races_dir.glob("*.parquet"))
all_laps = pd.concat([pd.read_parquet(f) for f in files], ignore_index=True)

print(len(files), "races")
print(all_laps.shape)

92 races
(83684, 35)


In [2]:
per_race = all_laps.groupby(["Year", "Round", "Event"]).size()
per_race.sort_values().head(10)

Year  Round  Event                   
2022  18     Japanese Grand Prix         370
2025  12     British Grand Prix          435
      1      Australian Grand Prix       547
2022  2      Saudi Arabian Grand Prix    588
      17     Singapore Grand Prix        617
      14     Belgian Grand Prix          624
2023  21     Las Vegas Grand Prix        639
2022  10     British Grand Prix          646
2023  12     Belgian Grand Prix          653
2024  23     Qatar Grand Prix            664
dtype: int64

In [3]:
all_laps["Compound"].value_counts()

Compound
HARD            38188
MEDIUM          30649
SOFT             9955
INTERMEDIATE     3987
None              364
nan               306
WET               235
Name: count, dtype: int64

In [4]:
print(len(files), "files")
all_laps.groupby("Year")["Round"].nunique()

92 files


Year
2022    22
2023    22
2024    24
2025    24
Name: Round, dtype: int64

In [5]:
for year, group in all_laps.groupby("Year"):
    rounds = set(group["Round"])
    expected = set(range(1, max(rounds) + 1))
    print(year, "missing:", sorted(expected - rounds))

2022 missing: []
2023 missing: []
2024 missing: []
2025 missing: []


In [6]:
all_laps.duplicated(["Year", "Round", "Driver", "LapNumber"]).sum()

np.int64(0)

In [7]:
key_cols = ["LapTimeSec", "TyreLife", "Compound", "LapNumber", "Stint", "Driver", "Team"]
all_laps[key_cols].isna().sum()

LapTimeSec      0
TyreLife      781
Compound        0
LapNumber       0
Stint         306
Driver          0
Team            0
dtype: int64

In [8]:
all_laps[["LapTimeSec", "TyreLife", "LapNumber"]].describe()

,LapTimeSec,TyreLife,LapNumber
count,83684.000000,82903.000000,83684.000000
mean,88.951262,15.285973,31.470783
std,10.893721,10.078806,17.921230
min,67.012000,1.000000,2.000000
25%,80.811000,8.000000,16.000000
50%,87.240000,13.000000,31.000000
75%,97.327250,21.000000,45.000000
max,149.321000,78.000000,78.000000


In [9]:
dry = all_laps[
    all_laps["Compound"].isin(["SOFT", "MEDIUM", "HARD"])
    & all_laps["TyreLife"].notna()
].copy()
print(len(all_laps), "→", len(dry))

83684 → 78633


In [10]:
stint_keys = ["Year", "Round", "Driver", "Stint"]

dry["StintMedian"] = (
    dry.groupby(stint_keys)["LapTimeSec"]
       .transform("median")
)

In [11]:
dry[["Year", "Event", "Driver", "Stint", "LapNumber", "LapTimeSec", "StintMedian"]].head(10)

,Year,Event,Driver,Stint,LapNumber,LapTimeSec,StintMedian
0,2022,Bahrain Grand Prix,VER,1.0,2.0,97.880,99.077
1,2022,Bahrain Grand Prix,VER,1.0,3.0,98.357,99.077
2,2022,Bahrain Grand Prix,VER,1.0,4.0,98.566,99.077
3,2022,Bahrain Grand Prix,VER,1.0,5.0,98.877,99.077
4,2022,Bahrain Grand Prix,VER,1.0,6.0,98.940,99.077
5,2022,Bahrain Grand Prix,VER,1.0,7.0,99.062,99.077
6,2022,Bahrain Grand Prix,VER,1.0,8.0,99.092,99.077
7,2022,Bahrain Grand Prix,VER,1.0,9.0,99.251,99.077
8,2022,Bahrain Grand Prix,VER,1.0,10.0,99.392,99.077
9,2022,Bahrain Grand Prix,VER,1.0,11.0,99.479,99.077


In [13]:
dry["SlowerPct"] = (
    (dry["LapTimeSec"] - dry["StintMedian"])
    / dry["StintMedian"]
    * 100
)
for threshold in [2, 3, 5, 7, 10]:
    n_slow = (dry["SlowerPct"] > threshold).sum()
    share = n_slow / len(dry) * 100
    print(f"slower than {threshold}%: {n_slow} laps ({share:.1f}%)")

slower than 2%: 2554 laps (3.2%)
slower than 3%: 1176 laps (1.5%)
slower than 5%: 400 laps (0.5%)
slower than 7%: 246 laps (0.3%)
slower than 10%: 156 laps (0.2%)


In [14]:
OUTLIER_THRESHOLD = 5

model_laps = dry[dry["SlowerPct"] <= OUTLIER_THRESHOLD].copy()

print("before: ", len(dry))
print("after:  ", len(model_laps))
print("removed:", len(dry) - len(model_laps))

before:  78633
after:   78233
removed: 400


In [15]:
model_laps.to_parquet("../data/processed/laps_model.parquet")
print("saved:", model_laps.shape)

saved: (78233, 37)
